# Retry Failed Topic Labels

This notebook scans all `topic_labels.csv` files for failed entries, then retries with **increasing temperature** until success or max temperature reached.

**Failure criteria:**
- `enriched_description` = `"No description available."`
- `label` starts with `"Topic_"` (fallback label)

**Retry strategy:** Start at temperature 0.3, increase by 0.2 each round up to 1.0 (max 4 attempts).

**Error logging:** Every failed attempt is logged with the raw LLM response and parse error for human review.

In [1]:
import os
import re
import json
import time
import pickle
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
from itertools import groupby
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## Configuration

In [ ]:
LIST_MODELS = ["lda", "dtm", "bertopic", "topicGpt"]
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../results")
LOG_DIR = Path("../../models/labeling/retry_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

# LLM Configuration
LLM_API_URL = "http://localhost:1234/v1/chat/completions"
LLM_MODEL = "mistralai/ministral-3-3b"
LLM_MAX_TOKENS = 4096

# Retry temperature schedule: escalate from 0.3 to 1.0
TEMPERATURE_SCHEDULE = [0.3, 0.5, 0.7, 1.0]

print(f"Models: {LIST_MODELS}")
print(f"Subjects: {LIST_SUBJECT}")
print(f"Temperature schedule: {TEMPERATURE_SCHEDULE}")
print(f"Error logs will be saved to: {LOG_DIR}")

Models: ['lda', 'topicGpt']
Subjects: ['cs', 'physics', 'math']
Temperature schedule: [0.3, 0.5, 0.7, 1.0]
Error logs will be saved to: ../../models/labeling/retry_logs


## LLM & Parsing Helpers

In [3]:
def call_llm(system_prompt: str, user_prompt: str, temperature: float, max_retries: int = 3) -> str:
    """Call LM Studio API with specific temperature and retry logic."""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": LLM_MAX_TOKENS,
    }
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                LLM_API_URL,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=120
            )
            resp.raise_for_status()
            data = resp.json()
            
            if "choices" in data:
                return data["choices"][0]["message"]["content"].strip()
            elif "content" in data:
                return data["content"].strip()
            elif "output" in data:
                return data["output"].strip()
            else:
                return str(data)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"    Network retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                return f"[LLM_ERROR] {e}"

def clean_and_parse_json(response: str) -> tuple:
    """Parse JSON from LLM response. Returns (parsed_dict, error_msg)."""
    if not response or response.startswith("[LLM_ERROR]"):
        return None, f"LLM returned error: {response}"
    
    text = re.sub(r"```json\s*|```", "", response).strip()
    
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1:
        return None, f"No JSON braces found in response. Raw: {response[:300]}"
    
    json_str = text[start:end+1]
    json_str = json_str.replace('\n', ' ').replace('\r', '')
    
    try:
        return json.loads(json_str), None
    except json.JSONDecodeError as e:
        error1 = str(e)
    
    # Fallback: regex extraction
    try:
        result = {}
        for field in ["label", "enriched_description"]:
            match = re.search(rf'"' + field + r'":\s*"(.*?)"', json_str, re.DOTALL)
            if match:
                result[field] = match.group(1).strip()
        if result:
            return result, None
    except Exception as e2:
        pass
    
    return None, f"JSON parse failed: {error1}. Raw JSON substring: {json_str[:300]}"

# Test LLM connection
test = call_llm("You are helpful.", "Say OK.", temperature=0.3)
print(f"LLM test: {test[:80]}")

LLM test: Got it! 😊 How can I assist you today?


## Scan All Files for Failures

In [4]:
def is_failed_row(row) -> tuple:
    """Check if a row has a failed label or description. Returns (is_failed, failure_types)."""
    failures = []
    label = str(row.get("label", ""))
    desc = str(row.get("enriched_description", ""))
    
    if re.match(r'^Topic_\d+$', label):
        failures.append("label_fallback")
    if desc == "No description available." or desc == "nan" or desc.strip() == "":
        failures.append("no_description")
    
    return (len(failures) > 0, failures)

# Scan all files
all_failures = []

for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        label_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        if not label_path.exists():
            print(f"  [SKIP] {model}/{subject}: file not found")
            continue
        
        df = pd.read_csv(label_path)
        failed_count = 0
        
        for idx, row in df.iterrows():
            failed, failure_types = is_failed_row(row)
            if failed:
                all_failures.append({
                    "model": model,
                    "subject": subject,
                    "topic_id": row["topic_id"],
                    "old_label": row["label"],
                    "old_description": str(row["enriched_description"])[:80],
                    "failure_types": failure_types
                })
                failed_count += 1
        
        total = len(df)
        status = "OK" if failed_count == 0 else f"{failed_count} failures"
        print(f"  {model}/{subject}: {total} topics, {status}")

print(f"\nTOTAL FAILURES TO RETRY: {len(all_failures)}")
failures_df = pd.DataFrame(all_failures)
if len(failures_df) > 0:
    print("\nFailure breakdown:")
    print(failures_df.groupby(["model", "subject"]).size().to_string())

  lda/cs: 50 topics, OK
  lda/physics: 50 topics, 1 failures
  lda/math: 50 topics, OK
  topicGpt/cs: 276 topics, OK
  topicGpt/physics: 188 topics, OK
  topicGpt/math: 124 topics, OK

TOTAL FAILURES TO RETRY: 1

Failure breakdown:
model  subject
lda    physics    1


## Prompts (same as original notebook)

In [5]:
LABEL_SYSTEM_PROMPT = """You are an expert academic topic analyst specializing in scientific literature.
Given a set of representative keywords from a topic discovered across multiple years of academic papers,
provide a concise label and a rich description for this topic.

OUTPUT RULES:
1. Return ONLY valid JSON: {"label": "...", "enriched_description": "..."}
2. The "label" must be 2-5 words, concise and descriptive.
3. The "enriched_description" must be 3-5 sentences describing the topic's scope, key methods, and applications in academic research.
4. Use PLAIN TEXT only. No markdown, no bolding (**), and no bullet points (-).
5. If you use quotes inside values, use 'single quotes' so the JSON doesn't break.
6. Keep the entire description on ONE SINGLE LINE. No newlines inside the JSON value."""

LABEL_USER_TEMPLATE = """Topic ID: {topic_id}
Subject Area: {subject}

Below are all the representative keywords for this topic, collected across multiple years of academic papers:

{all_words}

Based on these keywords, provide a concise label and a rich academic description for this topic.
Return ONLY valid JSON: {{"label": "...", "enriched_description": "..."}}"""

print("Prompts loaded.")

Prompts loaded.


## Retry Engine with Temperature Escalation

For each failed row:
1. Load original topic words from `topic_word_evolution.csv`
2. Try LLM call at temperature 0.3 -> 0.5 -> 0.7 -> 1.0
3. Stop as soon as a valid JSON response is parsed
4. Log **every** attempt with raw response + error cause

In [6]:
def load_topic_words_for_topic(model: str, subject: str, topic_id: int) -> str:
    """Load and deduplicate all top words across years for a specific topic."""
    path = BASE_DIR / model / "temporal" / subject / "topic_word_evolution.csv"
    df = pd.read_csv(path)
    topic_rows = df[df["topic_id"] == topic_id]
    
    all_words = []
    seen = set()
    for _, row in topic_rows.iterrows():
        words = [w.strip() for w in str(row["top_words"]).split(",")]
        for w in words:
            if w and w not in seen:
                all_words.append(w)
                seen.add(w)
    
    return ", ".join(all_words)

def retry_single_topic(model: str, subject: str, topic_id: int, failure_types: list) -> dict:
    """
    Retry a single failed topic with escalating temperature.
    Returns dict with FULL results and error log.
    """
    words_str = load_topic_words_for_topic(model, subject, topic_id)
    
    user_prompt = LABEL_USER_TEMPLATE.format(
        topic_id=topic_id,
        subject=subject,
        all_words=words_str
    )
    
    attempt_logs = []
    
    for temp in TEMPERATURE_SCHEDULE:
        raw_response = call_llm(LABEL_SYSTEM_PROMPT, user_prompt, temperature=temp)
        parsed, error_msg = clean_and_parse_json(raw_response)
        
        attempt_log = {
            "temperature": temp,
            "raw_response": raw_response[:500],
            "parse_error": error_msg,
            "success": False
        }
        
        if parsed:
            label = parsed.get("label", "")
            desc = parsed.get("enriched_description", "")
            
            # Validate: both fields must be non-empty and not fallback values
            label_ok = label and not re.match(r'^Topic_\d+$', label)
            desc_ok = desc and desc != "No description available." and len(desc) > 20
            
            if label_ok and desc_ok:
                attempt_log["success"] = True
                attempt_logs.append(attempt_log)
                return {
                    "success": True,
                    "label": label,
                    "enriched_description": desc,
                    "resolved_at_temp": temp,
                    "attempts": attempt_logs
                }
            else:
                reasons = []
                if not label_ok:
                    reasons.append(f"label invalid: '{label}'")
                if not desc_ok:
                    reasons.append(f"description invalid (len={len(desc) if desc else 0})")
                attempt_log["parse_error"] = f"Parsed but content invalid: {'; '.join(reasons)}"
        
        attempt_logs.append(attempt_log)
    
    # All temperatures failed
    return {
        "success": False,
        "label": None,
        "enriched_description": None,
        "resolved_at_temp": None,
        "attempts": attempt_logs
    }

print("Retry engine ready.")

Retry engine ready.


## Run Retries

In [7]:
# Process all failures
retry_results = []           # Summary for logging (preview only)
full_retry_data = {}         # (model, subject, topic_id) -> {label, enriched_description} FULL text
error_logs = []              # Detailed per-attempt logs
success_count = 0
fail_count = 0

sorted_failures = sorted(all_failures, key=lambda x: (x["model"], x["subject"]))

for (model, subject), group in groupby(sorted_failures, key=lambda x: (x["model"], x["subject"])):
    failures_list = list(group)
    print(f"\n{'='*60}")
    print(f"RETRYING: {model.upper()} / {subject.upper()} ({len(failures_list)} failures)")
    print(f"{'='*60}")
    
    for failure in tqdm(failures_list, desc=f"Retry {model}/{subject}"):
        topic_id = failure["topic_id"]
        result = retry_single_topic(model, subject, topic_id, failure["failure_types"])
        
        # Store FULL descriptions for CSV update
        if result["success"]:
            full_retry_data[(model, subject, topic_id)] = {
                "label": result["label"],
                "enriched_description": result["enriched_description"]
            }
        
        # Store summary for logging (preview only)
        retry_entry = {
            "model": model,
            "subject": subject,
            "topic_id": topic_id,
            "old_label": failure["old_label"],
            "old_description": failure["old_description"],
            "failure_types": ", ".join(failure["failure_types"]),
            "retry_success": result["success"],
            "new_label": result["label"],
            "new_description_preview": str(result["enriched_description"])[:200] if result["enriched_description"] else None,
            "resolved_at_temp": result["resolved_at_temp"]
        }
        retry_results.append(retry_entry)
        
        if result["success"]:
            success_count += 1
            print(f"  OK Topic {topic_id} fixed at temp={result['resolved_at_temp']}: {result['label']}")
        else:
            fail_count += 1
            print(f"  FAIL Topic {topic_id} STILL FAILED after all temperatures")
        
        # Log all attempts for this topic (for human review)
        for attempt in result["attempts"]:
            error_logs.append({
                "model": model,
                "subject": subject,
                "topic_id": topic_id,
                "temperature": attempt["temperature"],
                "success": attempt["success"],
                "parse_error": attempt["parse_error"],
                "raw_response_preview": attempt["raw_response"][:300]
            })

print(f"\n{'='*60}")
print(f"RETRY COMPLETE: {success_count} fixed, {fail_count} still failed out of {len(all_failures)} total")
print(f"{'='*60}")


RETRYING: LDA / PHYSICS (1 failures)


Retry lda/physics: 100%|██████████| 1/1 [00:03<00:00,  3.27s/it]

  OK Topic 9 fixed at temp=0.3: Bioinspired Catalytic Nanomaterials for Environmental and Biomedical Applications

RETRY COMPLETE: 1 fixed, 0 still failed out of 1 total


## Update CSV Files with Fixed Results

In [8]:
# Apply successful retries back to the CSV files
# Uses full_retry_data dict which stores complete (non-truncated) descriptions

updates_by_file = defaultdict(list)
for (model, subject, topic_id), data in full_retry_data.items():
    updates_by_file[(model, subject)].append((topic_id, data))

total_updated = 0
for (model, subject), updates in sorted(updates_by_file.items()):
    label_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
    df = pd.read_csv(label_path)
    
    for topic_id, data in updates:
        mask = df["topic_id"] == topic_id
        if mask.any():
            df.loc[mask, "label"] = data["label"]
            df.loc[mask, "enriched_description"] = data["enriched_description"]
    
    df.to_csv(label_path, index=False)
    total_updated += len(updates)
    print(f"  {model}/{subject}: updated {len(updates)} rows")

print(f"\nUpdated {len(updates_by_file)} CSV files with {total_updated} total fixes.")

  lda/physics: updated 1 rows

Updated 1 CSV files with 1 total fixes.


## Save Error Logs for Human Review

Three log files are saved:
1. **retry_error_log** - Every single attempt detail (raw response + parse error)
2. **retry_results** - Summary of each topic retry (success/fail)
3. **still_failed** - Topics that still need manual intervention

In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Full error logs (every attempt, including successful ones)
error_df = pd.DataFrame(error_logs)
error_log_path = LOG_DIR / f"retry_error_log_{timestamp}.csv"
error_df.to_csv(error_log_path, index=False)
print(f"Full error log: {error_log_path}")
print(f"  Entries: {len(error_df)}")

# 2. Summary of retry results
results_df = pd.DataFrame(retry_results)
results_path = LOG_DIR / f"retry_results_{timestamp}.csv"
results_df.to_csv(results_path, index=False)
print(f"\nRetry results: {results_path}")
print(f"  Entries: {len(results_df)}")

# 3. Still-failed entries (for manual intervention)
still_failed = results_df[results_df["retry_success"] == False]
if len(still_failed) > 0:
    still_failed_path = LOG_DIR / f"still_failed_{timestamp}.csv"
    still_failed.to_csv(still_failed_path, index=False)
    print(f"\nStill-failed: {still_failed_path}")
    print(f"  Count: {len(still_failed)}")
    print(f"\n  Breakdown:")
    print(still_failed.groupby(["model", "subject"]).size().to_string())
else:
    print("\n  All failures were resolved!")

Full error log: ../../models/labeling/retry_logs/retry_error_log_20260609_225621.csv
  Entries: 1

Retry results: ../../models/labeling/retry_logs/retry_results_20260609_225621.csv
  Entries: 1

  All failures were resolved!


## Summary Report

In [10]:
results_df = pd.DataFrame(retry_results)

print("=" * 60)
print("RETRY SUMMARY REPORT")
print("=" * 60)
print(f"\nTotal failures found: {len(all_failures)}")
print(f"Successfully fixed:   {success_count} ({100*success_count/max(len(all_failures),1):.1f}%)")
print(f"Still failed:         {fail_count}")

if len(results_df[results_df['retry_success']]) > 0:
    print(f"\nTemperature that resolved issues:")
    temp_dist = results_df[results_df['retry_success']]['resolved_at_temp'].value_counts().sort_index()
    for temp, count in temp_dist.items():
        print(f"  temp={temp}: {count} topics fixed")

print(f"\nLog files: {LOG_DIR}")
print(f"  - retry_error_log_{timestamp}.csv  (all attempt details for debugging)")
print(f"  - retry_results_{timestamp}.csv     (summary of each retry)")
if fail_count > 0:
    print(f"  - still_failed_{timestamp}.csv      (entries needing manual review)")

# Per-model breakdown
print(f"\nPer-model breakdown:")
for model in LIST_MODELS:
    model_results = results_df[results_df['model'] == model]
    if len(model_results) == 0:
        continue
    fixed = model_results['retry_success'].sum()
    total = len(model_results)
    print(f"  {model}: {fixed}/{total} fixed")

RETRY SUMMARY REPORT

Total failures found: 1
Successfully fixed:   1 (100.0%)
Still failed:         0

Temperature that resolved issues:
  temp=0.3: 1 topics fixed

Log files: ../../models/labeling/retry_logs
  - retry_error_log_20260609_225621.csv  (all attempt details for debugging)
  - retry_results_20260609_225621.csv     (summary of each retry)

Per-model breakdown:
  lda: 1/1 fixed


## Verify Updated Files

In [11]:
# Final verification: count remaining failures in all CSV files
print("Post-retry verification:")
print("=" * 60)

total_remaining = 0
for model in LIST_MODELS:
    for subject in LIST_SUBJECT:
        label_path = BASE_DIR / model / "temporal" / subject / "topic_labels.csv"
        if not label_path.exists():
            continue
        
        df = pd.read_csv(label_path)
        remaining = 0
        for _, row in df.iterrows():
            failed, _ = is_failed_row(row)
            if failed:
                remaining += 1
        
        total = len(df)
        status = "all good" if remaining == 0 else f"{remaining} still failed"
        print(f"  {model}/{subject}: {total} topics - {status}")
        total_remaining += remaining

print(f"\nTotal remaining failures: {total_remaining}")

Post-retry verification:
  lda/cs: 50 topics - all good
  lda/physics: 50 topics - all good
  lda/math: 50 topics - all good
  topicGpt/cs: 276 topics - all good
  topicGpt/physics: 188 topics - all good
  topicGpt/math: 124 topics - all good

Total remaining failures: 0
